# Лабораторная работа №2. Токенизация


In [ ]:
import re
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer


In [ ]:
rows = []
with ZipFile('data/news_science.zip') as archive:
    for filename in sorted(archive.namelist()):
        if filename.startswith('news_science/') and filename.endswith('.txt'):
            text = archive.read(filename).decode('utf-8').strip()
            genre = 'news' if '/news' in filename else 'science'
            rows.append({'file': Path(filename).name, 'text': text, 'genre': genre})

data = pd.DataFrame(rows)
print(data.groupby('genre').size())
data[['file', 'genre']].head()


In [ ]:
def tokenize_words(text):
    return re.findall(r'[а-яё]+', text.lower())

stop_words = set(Path('data/stopwords_ru.txt').read_text(encoding='utf-8').splitlines())
stop_words.update(['это', 'который', 'которая', 'которое', 'которые', 'которых'])
data['tokens'] = data['text'].apply(tokenize_words)
print(data.iloc[0]['tokens'][:20])


## 2.1. Облака слов без стоп-слов


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, genre, title in zip(axes, ['news', 'science'], ['Новости', 'Научные тексты']):
    frequencies = Counter()
    for tokens in data.loc[data['genre'] == genre, 'tokens']:
        frequencies.update(word for word in tokens if word not in stop_words)
    cloud = WordCloud(
        width=800, height=500, background_color='white', random_state=42,
        collocations=False,
    ).generate_from_frequencies(frequencies)
    ax.imshow(cloud, interpolation='bilinear')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 2.2. Частотные биграммы и триграммы


In [ ]:
def get_ngrams(tokens, n):
    return [' '.join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

for genre in ['news', 'science']:
    for n in [2, 3]:
        frequencies = Counter()
        for tokens in data.loc[data['genre'] == genre, 'tokens']:
            frequencies.update(get_ngrams(tokens, n))
        print(f'{genre}: топ-20 {n}-грамм')
        display(pd.DataFrame(frequencies.most_common(20), columns=['n-грамма', 'Частота']))


## 2.3. Ключевые слова двух новостей по TF-IDF


In [ ]:
news = data[data['genre'] == 'news'].reset_index(drop=True)
keyword_vectorizer = TfidfVectorizer(
    tokenizer=tokenize_words, token_pattern=None, lowercase=False,
    stop_words=sorted(stop_words),
)
news_vectors = keyword_vectorizer.fit_transform(news['text'])
feature_names = keyword_vectorizer.get_feature_names_out()

for i in [0, 1]:
    scores = news_vectors[i].toarray()[0]
    top_indices = scores.argsort()[::-1][:10]
    print(news.iloc[i]['file'])
    print(news.iloc[i]['text'][:350])
    display(pd.DataFrame({
        'Слово': feature_names[top_indices],
        'TF-IDF': scores[top_indices].round(4),
    }))


## 2.4. Классификация жанров: слова и BPE


In [ ]:
texts = data['tokens'].apply(' '.join)
X_train, X_test, y_train, y_test = train_test_split(
    texts, data['genre'], test_size=0.25, random_state=42, stratify=data['genre'],
)
print('Обучение:', len(X_train), 'текстов')
print('Проверка:', len(X_test), 'текстов')


In [ ]:
bpe = Tokenizer(BPE(unk_token='[UNK]'))
bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=2000, min_frequency=2, special_tokens=['[UNK]'],
    continuing_subword_prefix='##', show_progress=False,
)
bpe.train_from_iterator(X_train, trainer=trainer)

def tokenize_bpe(text):
    return bpe.encode(text).tokens

sample = ' '.join(X_test.iloc[0].split()[:8])
print('Слова:', tokenize_words(sample))
print('Части слов:', tokenize_bpe(sample))


In [ ]:
results = []
for name, tokenizer in [('Слова', tokenize_words), ('BPE', tokenize_bpe)]:
    vectorizer = TfidfVectorizer(
        tokenizer=tokenizer, token_pattern=None, lowercase=False,
    )
    train_vectors = vectorizer.fit_transform(X_train)
    test_vectors = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(train_vectors, y_train)
    predictions = model.predict(test_vectors)

    results.append({
        'Токенизатор': name,
        'Признаков': len(vectorizer.vocabulary_),
        'Accuracy': accuracy_score(y_test, predictions),
        'Macro-F1': f1_score(y_test, predictions, average='macro'),
    })
    print(name, '— матрица ошибок: строки — настоящий жанр, столбцы — предсказанный')
    display(pd.DataFrame(
        confusion_matrix(y_test, predictions, labels=['news', 'science']),
        index=['news', 'science'], columns=['news', 'science'],
    ))

pd.DataFrame(results).round(4)
